# Data sampling

Generate the citation-data coverage table for the fixed 400 seed papers from the local database. Collected text/author coverage is counted independently of model outputs. The final table row is the raw-input intersection: required text inputs and valid author lists on both papers. Publication dates do not determine input eligibility. Existing scores never determine cohort membership. Score availability and missing temporal inputs are reported separately in the diagnostic audit. No new seed papers are selected, no API calls are made, and PostgreSQL is never modified. Read-only queries are the primary source, and aggregate CSV/LaTeX files are output snapshots rather than raw-data caches.

The seed must be supplied by the `SEED` environment variable (or the existing local `.env`); no default is defined. Run cells in order from `src/` or the repository root using the project's database-enabled Python environment.


## 1. Configuration

Load database credentials and `SEED` through the environment without printing secrets. The existing `sample/seed_papers_final.xlsx` supplies the fixed seed IDs; its contents are not resampled or overwritten. Derived outputs use stable names under `sample/data_sampling/`.


In [1]:
from datetime import date, datetime, timezone
from pathlib import Path
import hashlib
import json
import os
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from dotenv import load_dotenv
from sqlalchemy import URL, create_engine, select
from sqlalchemy.engine import make_url
from sqlalchemy.orm import aliased
from sqlalchemy.pool import NullPool

SRC_DIR = Path.cwd().resolve()
if not (SRC_DIR / "postgres" / "model.py").is_file():
    SRC_DIR = SRC_DIR / "src"
if not (SRC_DIR / "postgres" / "model.py").is_file():
    raise RuntimeError("Run this notebook from src/ or the repository root.")
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
from postgres.model import Author, Citation, Paper, PaperAuthor

load_dotenv(SRC_DIR.parent / ".env", override=False)
load_dotenv(SRC_DIR / ".env", override=False)


def read_seed_setting(environment):
    raw = environment.get("SEED")
    if raw is None or not str(raw).strip():
        raise RuntimeError("SEED is required in the environment; no default is supplied.")
    try:
        value = int(raw)
    except (TypeError, ValueError):
        raise ValueError("SEED must be an integer.") from None
    if value < 0:
        raise ValueError("SEED must be nonnegative.")
    return value


SEED = read_seed_setting(os.environ)
SEED_FILE = SRC_DIR / "sample" / "seed_papers_final.xlsx"
OUTPUT_DIR = SRC_DIR / "sample" / "data_sampling"
SAVE_RESULTS = True
# SEED records the configured sampling seed. This notebook does not resample seeds.


## 2. Define the database reader

Check the fixed seed input, verify that every seed exists in PostgreSQL, and read citation and author inputs in one consistent read-only transaction. The query includes every stored citing-cited pair to these seeds, not just context-bearing pairs. Raw contexts, abstracts, dates, years, author records, and saved scoring fields are read in one snapshot. Saved scores are used only for readiness diagnostics, never for table counts or input-cohort selection. Records and identifiers remain in memory.


In [2]:
def load_fixed_seeds(path, expected_count=400):
    seeds = pd.read_excel(path, usecols=["id"])
    if seeds["id"].map(lambda x: isinstance(x, (bool, np.bool_))).any():
        raise ValueError("Seed IDs cannot be boolean values.")
    values = pd.to_numeric(seeds["id"], errors="raise")
    if (values.isna().any() or values.le(0).any()
            or values.mod(1).ne(0).any() or values.duplicated().any()):
        raise ValueError("Seed IDs must be distinct positive integers.")
    if len(values) != expected_count:
        raise ValueError(f"Expected {expected_count} fixed seeds.")
    return values.astype("int64").tolist()


def read_coverage_inputs(seed_ids):
    """Read inputs in one consistent, read-only PostgreSQL snapshot."""
    if not seed_ids or len(set(seed_ids)) != len(seed_ids):
        raise ValueError("Seed IDs must be nonempty and unique.")
    if os.getenv("DATABASE_URL"):
        url = make_url(os.environ["DATABASE_URL"]).set(drivername="postgresql+psycopg")
    else:
        if not os.getenv("POSTGRES_USER") or not os.getenv("POSTGRES_DB"):
            raise RuntimeError("Set POSTGRES_USER and POSTGRES_DB, or DATABASE_URL.")
        url = URL.create(
            "postgresql+psycopg", username=os.environ["POSTGRES_USER"],
            password=os.getenv("POSTGRES_PASS") or os.getenv("POSTGRES_PASSWORD"),
            host=os.getenv("POSTGRES_HOST", "localhost"),
            port=int(os.getenv("POSTGRES_PORT", "5432")),
            database=os.environ["POSTGRES_DB"],
        )
    engine = create_engine(
        url, poolclass=NullPool, isolation_level="REPEATABLE READ",
        connect_args={
            "connect_timeout": 10,
            "options": "-c default_transaction_read_only=on "
                       "-c statement_timeout=120000 -c lock_timeout=5000",
        },
    )
    citing, cited = aliased(Paper), aliased(Paper)
    query = (
        select(
            Citation.citing_corpus_id, Citation.cited_corpus_id, Citation.contexts,
            Citation.cross_score, Citation.llm_score,
            Citation.abstract_cross_score, Citation.abstract_llm_score,
            Citation.author_independence,
            citing.abstract.label("citing_abstract"),
            citing.publication_date.label("citing_date"),
            citing.year.label("citing_year"),
            cited.abstract.label("cited_abstract"),
            cited.publication_date.label("cited_date"),
            cited.year.label("cited_year"),
        )
        .select_from(Citation)
        .outerjoin(citing, citing.corpus_id == Citation.citing_corpus_id)
        .outerjoin(cited, cited.corpus_id == Citation.cited_corpus_id)
        .where(Citation.cited_corpus_id.in_(seed_ids))
        .order_by(Citation.cited_corpus_id, Citation.citing_corpus_id)
    )
    try:
        with engine.connect() as connection, connection.begin():
            present = set(connection.scalars(
                select(Paper.corpus_id).where(Paper.corpus_id.in_(seed_ids))
            ))
            if len(present) != len(seed_ids):
                raise ValueError("Some fixed seeds are absent from the database.")
            result = connection.execute(query)
            citations = pd.DataFrame(result.fetchall(), columns=list(result.keys()))
            paper_ids = sorted(set(seed_ids) | set(citations["citing_corpus_id"]))
            author_query = (
                select(
                    PaperAuthor.paper_id, PaperAuthor.author_id,
                    PaperAuthor.author_order, Author.name.label("author_name"),
                )
                .outerjoin(Author, Author.author_id == PaperAuthor.author_id)
                .where(PaperAuthor.paper_id.in_(paper_ids))
                .order_by(PaperAuthor.paper_id, PaperAuthor.author_order)
            )
            result = connection.execute(author_query)
            authors = pd.DataFrame(result.fetchall(), columns=list(result.keys()))
    except ValueError:
        raise
    except Exception as error:
        # Never print credentials, connection URLs, or SQL parameters.
        raise RuntimeError(
            f"Read-only database query failed ({type(error).__name__}). "
            "Check the connection and schema locally."
        ) from None
    finally:
        engine.dispose()
    return citations, authors


## 3. Define collected-data coverage

Partition stored pairs into usable context, no context with citing abstract, neither input, and malformed context structures. Context plus citing abstract is a subset of the context row, not another input category. Required text inputs mean a cited-paper abstract and usable context or, only when context is unavailable, a citing-paper abstract.

Count valid author lists on both papers across the entire stored pool, together with self-citations in that author-valid pool. The final joint-input row intersects the text-ready cohort with valid author lists on both papers. Valid lists require positive unique author IDs, nonempty names, and unique contiguous 0/1-based author order. Neither publication dates nor computed scores determine this intersection.

Both citing_date and cited_date come from Paper.publication_date through their respective citing/cited joins. Dates and years remain diagnostic only, with no year-only or cited-date substitution for a full citing date. The context-bearing and no-context/citing-abstract groups are disjoint; their union forms the text-ready cohort when the cited abstract is present. Cases lacking both text inputs remain in the audit but are omitted from the manuscript table. The audit separately checks semantic-score availability, recorded independence scores, and missing temporal inputs. Input eligibility does not establish that a temporally valid independence score can be computed. No missing score is replaced by 1, and no scores are recomputed.


In [3]:
def has_text(value):
    return isinstance(value, str) and bool(value.strip())


def has_publication_date(value):
    """Recognize a full calendar date; never turn a year into a day."""
    if isinstance(value, date):
        return not pd.isna(value)
    if isinstance(value, str):
        text = value.strip()
        if len(text) != 10 or text[4] != "-" or text[7] != "-":
            return False
        try:
            date.fromisoformat(text)
            return True
        except ValueError:
            return False
    return False


def inspect_context(value):
    """Return (available, valid); do not reinterpret malformed JSON as missing."""
    if value is None:
        return False, True
    if isinstance(value, str):
        return has_text(value), True
    if isinstance(value, dict):
        for key in ("text", "context", "sentence"):
            if key in value:
                return inspect_context(value[key])
        return False, False
    if isinstance(value, list):
        items = [inspect_context(item) for item in value]
        return any(x[0] for x in items), all(x[1] for x in items)
    return False, False


def publication_date_coverage(citations):
    """Compare dates on the two Paper joins; count citation pairs, not unique papers."""
    result = []
    for role in ("citing", "cited"):
        dated = citations[f"{role}_date"].map(has_publication_date).astype(bool)
        years = pd.to_numeric(citations[f"{role}_year"], errors="coerce")
        year_ready = (years.between(1, 9999) & years.mod(1).eq(0)
                      & ~citations[f"{role}_year"].map(
                          lambda x: isinstance(x, (bool, np.bool_))))
        result.append(dict(
            role=role, join_key=f"Citation.{role}_corpus_id",
            date_source="Paper.publication_date", total_pairs=len(citations),
            pairs_with_date=int(dated.sum()), pairs_with_year=int(year_ready.sum()),
        ))
    return pd.DataFrame(result)


def usable_author_list(group):
    ids = pd.to_numeric(group["author_id"], errors="coerce")
    order = pd.to_numeric(group["author_order"], errors="coerce")
    if (group["author_id"].map(lambda x: isinstance(x, (bool, np.bool_))).any()
            or group["author_order"].map(lambda x: isinstance(x, (bool, np.bool_))).any()
            or ids.isna().any() or ids.le(0).any() or ids.mod(1).ne(0).any()
            or ids.duplicated().any() or order.isna().any()
            or order.mod(1).ne(0).any() or order.duplicated().any()
            or not group["author_name"].map(has_text).all()):
        return False
    start = int(order.min())
    return start in (0, 1) and sorted(order.astype(int)) == list(
        range(start, start + len(group))
    )


def compute_coverage(citations, authors, seed_ids):
    """Count collected fields only; model outputs are not required or inspected."""
    required = {
        "citing_corpus_id", "cited_corpus_id", "contexts", "citing_abstract",
        "cited_abstract", "citing_date", "citing_year", "cited_date", "cited_year",
    }
    if required - set(citations.columns):
        raise ValueError("Citation input is missing required collected-data columns.")
    if not {"paper_id", "author_id", "author_order", "author_name"} <= set(authors):
        raise ValueError("Author input is missing required columns.")
    if len(set(seed_ids)) != len(seed_ids):
        raise ValueError("Duplicate fixed seed IDs.")
    if citations.duplicated(["citing_corpus_id", "cited_corpus_id"]).any():
        raise ValueError("Duplicate citation pairs.")
    if not citations["cited_corpus_id"].isin(seed_ids).all():
        raise ValueError("Query contains a cited paper outside the fixed seed set.")

    statuses = citations["contexts"].map(inspect_context)
    context_valid = statuses.map(lambda x: x[1]).astype(bool)
    context_available = statuses.map(lambda x: x[0]).astype(bool)
    citing_available = citations["citing_abstract"].map(has_text).astype(bool)
    cited_available = citations["cited_abstract"].map(has_text).astype(bool)
    context = context_valid & context_available
    both_inputs = context & citing_available
    abstract_only = context_valid & ~context_available & citing_available
    neither = context_valid & ~context_available & ~citing_available
    malformed = ~context_valid
    text_ready = cited_available & (context | abstract_only)

    # Direct self-citation detection needs only usable IDs, not names/order/dates.
    author_ids = {}
    validated_papers = set()
    for paper_id, group in authors.groupby("paper_id"):
        values = pd.to_numeric(group["author_id"], errors="coerce")
        valid = (np.isfinite(values) & values.gt(0) & values.mod(1).eq(0)
                 & ~group["author_id"].map(lambda x: isinstance(x, (bool, np.bool_))))
        author_ids[int(paper_id)] = set(values.loc[valid].astype(int))
        if usable_author_list(group):
            validated_papers.add(int(paper_id))
    papers_with_records = set(authors["paper_id"])
    both_author_records = (
        citations["cited_corpus_id"].isin(papers_with_records)
        & citations["citing_corpus_id"].isin(papers_with_records)
    )
    author_records = text_ready & both_author_records
    author_lists = (
        citations["cited_corpus_id"].isin(validated_papers)
        & citations["citing_corpus_id"].isin(validated_papers)
    )
    shared_authors = pd.Series([
        bool(author_ids.get(int(citing), set()) & author_ids.get(int(cited), set()))
        for citing, cited in zip(citations["citing_corpus_id"], citations["cited_corpus_id"])
    ], index=citations.index, dtype=bool)
    self_citations = author_records & shared_authors
    citing_dated = citations["citing_date"].map(has_publication_date).astype(bool)
    dated_author_lists = author_lists & citing_dated
    dated_author_inputs = text_ready & dated_author_lists
    n_pool, n_text = len(citations), int(text_ready.sum())

    if int(context.sum() + abstract_only.sum() + neither.sum() + malformed.sum()) != n_pool:
        raise AssertionError("Input branches do not reconcile with the pool.")
    if int((text_ready & context).sum() + (text_ready & abstract_only).sum()) != n_text:
        raise AssertionError("Text-input branches do not reconcile.")

    rows = []
    def add(group, metric, label, count, denominator, denominator_count, note=""):
        rows.append(dict(
            group=group, metric=metric, label=label, count=int(count),
            denominator=denominator, denominator_count=int(denominator_count),
            latex_label=label, note=note,
        ))

    add("Input coverage", "stored_pairs", "Stored citation pairs",
        n_pool, "stored_pairs", n_pool)
    add("Input coverage", "context_inputs", "With usable citation context",
        context.sum(), "stored_pairs", n_pool)
    add("Input coverage", "context_and_citing_abstract", "With context and citing abstract",
        both_inputs.sum(), "context_inputs", context.sum(),
        note="Subset of usable-context pairs; not an additional input category.")
    add("Input coverage", "abstract_only_inputs", "Without context; citing abstract available",
        abstract_only.sum(), "stored_pairs", n_pool)
    add("Input coverage", "neither_input", "Neither context nor citing abstract",
        neither.sum(), "stored_pairs", n_pool)
    if malformed.any():
        add("Input coverage", "malformed_context", "Invalid context structure",
            malformed.sum(), "stored_pairs", n_pool)
    add("Input coverage", "input_ready_pairs", "With required text inputs",
        n_text, "stored_pairs", n_pool,
        note="Cited abstract plus usable context or, without context, citing abstract; no model outputs.")
    add("Author coverage", "valid_author_list_pairs", "With valid author lists",
        author_lists.sum(), "stored_pairs", n_pool,
        note="Both papers have valid author IDs, names and contiguous order; no text/score filter.")
    add("Author coverage", "self_citations_valid_author_pairs",
        "Self-citations among author-valid pairs", (author_lists & shared_authors).sum(),
        "valid_author_list_pairs", author_lists.sum(),
        note="Shared author IDs within the full author-valid pool; no text or score filter.")
    add("Joint input coverage", "joint_input_pairs", "Text-ready pairs with valid author lists",
        (text_ready & author_lists).sum(), "input_ready_pairs", n_text,
        note="Intersection of required text inputs and valid author lists on both papers; "
             "no date or score filter.")
    coverage = pd.DataFrame(rows)

    audit_rows = []
    def audit(metric, count, denominator, size, unit="citation_pairs"):
        audit_rows.append(dict(metric=metric, count=int(count),
                               denominator=denominator, denominator_count=int(size), unit=unit))
    audit("fixed_seed_papers", len(seed_ids), "fixed_seed_papers", len(seed_ids), "seed_papers")
    for row in rows:
        audit(row["metric"], row["count"], row["denominator"], row["denominator_count"])
    for row in publication_date_coverage(citations).itertuples(index=False):
        audit(f"{row.role}_publication_date_in_pool", row.pairs_with_date, "stored_pairs", n_pool)
        audit(f"{row.role}_publication_year_in_pool", row.pairs_with_year, "stored_pairs", n_pool)
    audit("missing_cited_abstract", (~cited_available).sum(), "stored_pairs", n_pool)
    audit("input_ready_context_pairs", (text_ready & context).sum(), "input_ready_pairs", n_text)
    audit("input_ready_abstract_only_pairs", (text_ready & abstract_only).sum(),
          "input_ready_pairs", n_text)
    audit("input_ready_author_records", author_records.sum(), "input_ready_pairs", n_text)
    audit("self_citations_in_author_record_pairs", self_citations.sum(),
          "input_ready_author_records", author_records.sum())
    audit("pool_with_author_records", both_author_records.sum(), "stored_pairs", n_pool)
    audit("pool_with_usable_author_lists", author_lists.sum(), "stored_pairs", n_pool)
    audit("self_citations_in_pool", (both_author_records & shared_authors).sum(),
          "pool_with_author_records", both_author_records.sum())
    audit("input_ready_with_usable_author_lists", (text_ready & author_lists).sum(),
          "input_ready_author_records", author_records.sum())
    audit("text_author_pairs_missing_citing_date",
          (text_ready & author_lists & ~citing_dated).sum(),
          "input_ready_with_usable_author_lists", (text_ready & author_lists).sum())
    audit("input_ready_missing_author_records", (text_ready & ~both_author_records).sum(),
          "input_ready_pairs", n_text)
    audit("dated_valid_author_pairs", dated_author_lists.sum(),
          "valid_author_list_pairs", author_lists.sum())
    audit("dated_author_inputs", dated_author_inputs.sum(),
          "input_ready_author_records", author_records.sum())
    audit("self_citations_in_dated_subset", (dated_author_inputs & shared_authors).sum(),
          "dated_author_inputs", dated_author_inputs.sum())
    represented = citations.loc[text_ready, "cited_corpus_id"].nunique()
    audit("seeds_with_required_text_inputs", represented,
          "fixed_seed_papers", len(seed_ids), "seed_papers")
    # Keep the excluded-input count in the audit, not the displayed manuscript table.
    coverage = coverage.loc[~coverage["metric"].eq("neither_input")].reset_index(drop=True)
    return coverage, pd.DataFrame(audit_rows)

def append_score_readiness_audit(coverage, audit, citations, authors):
    """Audit missing scores without excluding any raw-input-eligible pair."""
    def unit_score(series):
        values = pd.to_numeric(series, errors="coerce")
        not_boolean = ~series.map(lambda x: isinstance(x, (bool, np.bool_)))
        return values.where(np.isfinite(values) & values.between(0, 1) & not_boolean)

    status = citations["contexts"].map(inspect_context)
    context = status.map(lambda item: item[0])
    text_ready = (
        status.map(lambda item: item[1])
        & citations["cited_abstract"].map(has_text)
        & (context | citations["citing_abstract"].map(has_text))
    )
    valid_groups = {
        int(paper_id): group for paper_id, group in authors.groupby("paper_id")
        if usable_author_list(group)
    }
    joint = (
        text_ready & citations["citing_corpus_id"].isin(valid_groups)
        & citations["cited_corpus_id"].isin(valid_groups)
    )
    n_joint = int(joint.sum())
    expected = int(coverage.set_index("metric").loc["joint_input_pairs", "count"])
    if n_joint != expected:
        raise AssertionError("Readiness audit must use the complete raw-input intersection.")
    cross = unit_score(citations["cross_score"]).where(
        context, unit_score(citations["abstract_cross_score"])
    )
    llm = unit_score(citations["llm_score"]).where(
        context, unit_score(citations["abstract_llm_score"])
    )
    independence = unit_score(citations["author_independence"])
    missing_semantic = joint & (cross.isna() | llm.isna())
    missing_independence = joint & independence.isna()
    ids = {
        paper_id: set(pd.to_numeric(group["author_id"]).astype(int))
        for paper_id, group in valid_groups.items()
    }
    shared = pd.Series([
        bool(ids.get(int(citing), set()) & ids.get(int(cited), set()))
        for citing, cited in citations[["citing_corpus_id", "cited_corpus_id"]]
            .itertuples(index=False, name=None)
    ], index=citations.index)
    has_time = citations["citing_date"].map(has_publication_date).astype(bool)
    checks = {
        "joint_inputs_with_valid_semantic_scores": joint & cross.notna() & llm.notna(),
        "joint_inputs_score_complete": joint & cross.notna() & llm.notna() & independence.notna(),
        "joint_inputs_missing_semantic_scores": missing_semantic,
        "joint_inputs_missing_independence": missing_independence,
        "pending_independence_direct_overlap_ready": missing_independence & shared,
        "pending_independence_temporally_ready": missing_independence & ~shared & has_time,
        "pending_independence_missing_full_citing_date": missing_independence & ~shared & ~has_time,
        "joint_inputs_context": joint & context,
        "joint_inputs_abstract_only": joint & ~context,
    }
    rows = [
        dict(metric=metric, count=int(mask.sum()), denominator="joint_input_pairs",
             denominator_count=n_joint, unit="citation_pairs")
        for metric, mask in checks.items()
    ]
    rows.append(dict(
        metric="joint_input_seed_papers",
        count=int(citations.loc[joint, "cited_corpus_id"].nunique()),
        denominator="fixed_seed_papers",
        denominator_count=int(audit.set_index("metric").loc["fixed_seed_papers", "count"]),
        unit="seed_papers",
    ))
    return coverage, pd.concat([audit, pd.DataFrame(rows)], ignore_index=True)


## 4. Define aggregate exports

Format the coverage table as a short-caption, two-column LaTeX booktabs table. CSV exports include each row's denominator and definition; the diagnostic CSV records exclusions and branch counts. These helpers do not write files until the final export cell runs.


In [4]:
def latex_coverage_table(coverage):
    """Render aggregate counts as a compact booktabs table; no source identifiers."""
    lines = [
        r"\begin{table}",
        r"    \centering",
        r"    \caption{Citation-data coverage.}",
        r"    \label{tab:citation-coverage}",
        r"    \small",
        r"    \begin{tabular}{@{}p{0.80\columnwidth}r@{}}",
        r"        \toprule",
        r"        Citation cohort & Pairs \\",
        r"        \midrule",
    ]
    previous_group = None
    for row in coverage.itertuples(index=False):
        if row.group != previous_group:
            if previous_group is not None:
                lines.append(r"        \midrule")
            lines.append(
                r"        \multicolumn{2}{@{}l}{\textit{" + row.group + r"}} \\"
            )
            previous_group = row.group
        lines.append("        " + row.latex_label + f" & {int(row.count):,}" + r" \\")
    lines.extend([
        r"        \bottomrule",
        r"    \end{tabular}",
        r"\end{table}",
    ])
    return "\n".join(lines) + "\n"


def save_coverage_results(coverage, audit, latex_table, output_dir, metadata):
    """Save aggregate result snapshots only; overwrite these derived files."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    coverage.to_csv(output_dir / "citation_coverage.csv", index=False)
    audit.to_csv(output_dir / "citation_coverage_audit.csv", index=False)
    (output_dir / "citation_coverage.tex").write_text(latex_table, encoding="utf-8")
    with (output_dir / "metadata.json").open("w", encoding="utf-8") as stream:
        json.dump(metadata, stream, indent=2, allow_nan=False)


## 5. Read the current database snapshot

Load the fixed seeds and stored citation inputs. This is the only database-reading execution cell. Rerunning later calculation, display, and formatting cells reuses these in-memory inputs; no model calls are needed.


In [5]:
seed_ids = load_fixed_seeds(SEED_FILE)
citations, authors = read_coverage_inputs(seed_ids)
database_query_utc = datetime.now(timezone.utc).isoformat(timespec="seconds")
if citations.empty:
    raise RuntimeError("No stored citations to the fixed seeds; check the database before reporting results.")
print(f"Loaded {len(seed_ids):,} fixed seeds and {len(citations):,} stored citation pairs (read-only).")


Loaded 400 fixed seeds and 9,655 stored citation pairs (read-only).


## 6. Generate the manuscript coverage table

Display counts computed from collected text and author fields with their denominators. Author-valid and self-citation rows use the entire stored pool; the final row intersects text availability with valid author lists on both papers. Publication dates and existing scores do not filter this input cohort. Temporal completeness and score readiness are reported separately in the audit. Rerun the database-reading cell after changing definitions; counts are never hardcoded.


In [6]:
coverage_table, coverage_audit = compute_coverage(citations, authors, seed_ids)
coverage_table, coverage_audit = append_score_readiness_audit(
    coverage_table, coverage_audit, citations, authors,
)
display(coverage_table[["group", "label", "count", "denominator", "denominator_count"]])


,group,label,count,denominator,denominator_count
0,Input coverage,Stored citation pairs,9655,stored_pairs,9655
1,Input coverage,With usable citation context,4781,stored_pairs,9655
2,Input coverage,With context and citing abstract,4243,context_inputs,4781
3,Input coverage,Without context; citing abstract available,3206,stored_pairs,9655
4,Input coverage,With required text inputs,7987,stored_pairs,9655
5,Author coverage,With valid author lists,4852,stored_pairs,9655
6,Author coverage,Self-citations among author-valid pairs,458,valid_author_list_pairs,4852
7,Joint input coverage,Text-ready pairs with valid author lists,4818,input_ready_pairs,7987


## 7. Compare publication-date coverage

Both dates come from `Paper.publication_date`, joined using the indicated citing or cited Corpus ID. Counts below refer to citation pairs in the stored pool, not distinct papers. A cited-paper date cannot substitute for a missing citing-paper date; a year is not a full date. This is a raw-metadata diagnostic, not a model-output filter.


In [7]:
date_coverage = publication_date_coverage(citations)
display(date_coverage)
author_rows = coverage_table.loc[coverage_table["group"].eq("Author coverage")]
display(author_rows[["label", "count", "denominator", "denominator_count"]])


,role,join_key,date_source,total_pairs,pairs_with_date,pairs_with_year
0,citing,Citation.citing_corpus_id,Paper.publication_date,9655,1,1
1,cited,Citation.cited_corpus_id,Paper.publication_date,9655,9352,9655


,label,count,denominator,denominator_count
5,With valid author lists,4852,stored_pairs,9655
6,Self-citations among author-valid pairs,458,valid_author_list_pairs,4852


## 8. Display detailed collected-data checks

Show missing cited abstracts, context/abstract branches, author-record and validated-list availability before and after the text-input check, direct author overlap, publication dates/years, and represented seed counts. Only aggregate counts are displayed, including separate score-readiness diagnostics; no per-citation model outputs, source identifiers, or source text are displayed. Malformed contexts remain visible.


In [8]:
display(coverage_audit)


,metric,count,denominator,denominator_count,unit
0,fixed_seed_papers,400,fixed_seed_papers,400,seed_papers
1,stored_pairs,9655,stored_pairs,9655,citation_pairs
2,context_inputs,4781,stored_pairs,9655,citation_pairs
3,context_and_citing_abstract,4243,context_inputs,4781,citation_pairs
4,abstract_only_inputs,3206,stored_pairs,9655,citation_pairs
5,neither_input,1668,stored_pairs,9655,citation_pairs
6,input_ready_pairs,7987,stored_pairs,9655,citation_pairs
7,valid_author_list_pairs,4852,stored_pairs,9655,citation_pairs
8,self_citations_valid_author_pairs,458,valid_author_list_pairs,4852,citation_pairs
9,joint_input_pairs,4818,input_ready_pairs,7987,citation_pairs


## 9. Generate LaTeX table text

Display collected text/author coverage after checking counts. The caption is Citation-data coverage; no extra note is appended below the table. Cases lacking both context and citing abstract, as well as date-completeness counts, remain in the audit rather than the manuscript table. The final Joint input coverage row reports the raw text/author intersection without date- or score-based exclusion. Input eligibility is distinct from score readiness and temporally validated independence. Neither historical experiment cohorts nor expert subsets are reconstructed here.


In [9]:
coverage_latex = latex_coverage_table(coverage_table)
print(coverage_latex)


\begin{table}
    \centering
    \caption{Citation-data coverage.}
    \label{tab:citation-coverage}
    \small
    \begin{tabular}{@{}p{0.80\columnwidth}r@{}}
        \toprule
        Citation cohort & Pairs \\
        \midrule
        \multicolumn{2}{@{}l}{\textit{Input coverage}} \\
        Stored citation pairs & 9,655 \\
        With usable citation context & 4,781 \\
        With context and citing abstract & 4,243 \\
        Without context; citing abstract available & 3,206 \\
        With required text inputs & 7,987 \\
        \midrule
        \multicolumn{2}{@{}l}{\textit{Author coverage}} \\
        With valid author lists & 4,852 \\
        Self-citations among author-valid pairs & 458 \\
        \midrule
        \multicolumn{2}{@{}l}{\textit{Joint input coverage}} \\
        Text-ready pairs with valid author lists & 4,818 \\
        \bottomrule
    \end{tabular}
\end{table}


## 10. Save aggregate snapshots

With `SAVE_RESULTS=True`, save `citation_coverage.csv`, `citation_coverage_audit.csv`, `citation_coverage.tex`, and `metadata.json`. Metadata records query time, the configured environment seed, fixed-input and code hashes, filtering rules, and output paths. Repeat runs replace these derived files; original inputs are unchanged. No article IDs, author names, abstracts, contexts, per-citation scores, credentials, or SQL data dumps are exported.

These outputs document the current database snapshot; they do not guarantee that a later live API query reconstructs the same historical records.


In [10]:
if SAVE_RESULTS:
    source = json.loads((SRC_DIR / "data_sampling.ipynb").read_text(encoding="utf-8"))
    notebook_code = "\n\n".join(
        "".join(cell["source"]) for cell in source["cells"] if cell["cell_type"] == "code"
    )
    metadata = {
        "source": "Local PostgreSQL records collected through Semantic Scholar",
        "database_query_utc": database_query_utc,
        "seed_input": "sample/seed_papers_final.xlsx",
        "seed_input_sha256": hashlib.sha256(SEED_FILE.read_bytes()).hexdigest(),
        "notebook_code_sha256": hashlib.sha256(notebook_code.encode("utf-8")).hexdigest(),
        "configured_seed": SEED,
        "seed_source": "SEED environment variable; no default",
        "randomization": "None; fixed seed-paper input is read without resampling",
        "fixed_seed_count": len(seed_ids),
        "stored_pair_count": len(citations),
        "query_scope": "All stored citation pairs to the fixed cited seed IDs",
        "query_mode": "Read-only repeatable-read transaction; no API or model calls",
        "text_input_policy": "Cited abstract required plus context, or citing abstract only without context",
        "model_output_policy": "Saved scores used only for readiness audit; no cohort selection by scores",
        "author_policy": "Valid author lists across all stored pairs: positive unique IDs, nonempty "
                         "names, unique contiguous order; joint cohort intersects text availability without a date filter",
        "publication_date_policy": "Full citing/cited dates and years reported separately as diagnostics; "
                                   "no year-only or cited-date substitution, date imputation, or historical graph validation",
        "self_citation_rule": "Shared IDs within the full valid-author-list pool; no text or score filter",
        "date_filter": "None; publication-date completeness is diagnostic only",
        "denominators": "Recorded per row; author-valid pool is unconditional on text; joint row is intersection",
        "latex_table_policy": "Collected text and author coverage only; neither-input and date-qualified "
                              "counts remain diagnostic; final joint row requires text and valid author lists only",
        "outputs": ["citation_coverage.csv", "citation_coverage_audit.csv",
                    "citation_coverage.tex", "metadata.json"],
        "export_scope": "Aggregate counts and provenance only; no article IDs, author "
                        "names, abstracts, contexts, per-citation scores, or credentials",
    }
    save_coverage_results(
        coverage_table, coverage_audit, coverage_latex, OUTPUT_DIR, metadata,
    )
    print("Saved aggregate CSV tables, LaTeX table, and metadata to sample/data_sampling/.")
else:
    print("File export disabled (SAVE_RESULTS=False).")


Saved aggregate CSV tables, LaTeX table, and metadata to sample/data_sampling/.
